In [ ]:
# This stuff only actually gets used in this cell:

from matplotlib import pyplot as plt
from matplotlib import cm
from matplotlib import colors as mcolors

# Use the style file defined in the repository route
plt.style.use('../../paper.mplstyle')

# This is the folder where results plots will be saved:
paper_plots_dir = '../../paper/images/inpainting_results'

# The common and plotting utils modules are in the parent
# `search` directory
# Here we add the `search` directory to the python path so we can
# do a local import

import sys
import os

parent_dir = os.path.abspath("..")

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)


# Everything plotted in this notebook is done using these helper modules
import plotting_utils
from common_utils import get_results_inpainting as get_results, get_results_zero_latency

In [ ]:
times_before = [0.5, 1, 4, 7, 14]

results_estimate_filtered = get_results(
    f"results/data_runs_psd_estimate.hdf",
    filter_times_before=times_before
)
results_model_filtered = get_results(
    f"results/data_runs_psd_model.hdf",
    filter_times_before=times_before
)

results_estimate_all = get_results(
    f"results/data_runs_psd_estimate.hdf"
)
results_model_all = get_results(
    f"results/data_runs_psd_model.hdf"
)


In [ ]:
# Get the truth times
import ldc.io.hdf5 as hdfio

input_data = '../../datasets/LDC2_sangria_hm_training.hdf'
mbhb_sky = hdfio.load_array(input_data, name="sky/mbhb/cat")

truth_times_s = mbhb_sky[0]['CoalescenceTime']

In [ ]:
fig_filtered, _ = plotting_utils.plot_around_time(
    results_one=results_model_filtered,
    results_two=results_estimate_filtered,
    truth_times=truth_times_s,
    times_before=times_before,
    label_one='PSD model',
    label_two='Estimated PSD',
    legend_loc='upper right'
)
fig_filtered.savefig(f'plots/signals_results_filtered_by_time.pdf')


In [ ]:
fig_all, _ = plotting_utils.plot_around_time(
    results_one=results_model_all,
    results_two=results_estimate_all,
    truth_times=truth_times_s,
    times_before=times_before,
    label_one='PSD model',
    label_two='Estimated PSD'
)
fig_all.savefig(f'plots/signals_results_all_forecast.pdf')

In [ ]:
fig_all, _ = plotting_utils.plot_around_time(
    results_one=results_model_all,
    results_two=results_estimate_all,
    truth_times=truth_times_s,
    times_before=times_before,
    label_one='PSD model',
    label_two='Estimated PSD',
    predicted_time=False
)
fig_all.savefig(f'plots/signals_results_all_end_time.pdf')

In [ ]:
for i, truth_time_s in enumerate(truth_times_s):
    fig_model, _ = plotting_utils.plot_around_time(
        results_one=results_model_filtered,
        results_two=results_estimate_filtered,
        signal_truth_time=truth_time_s,
        truth_times=truth_times_s,
        start_time_offset=-20,
        end_time_offset=20,
        times_before=times_before,
        signal_number=i,
        label_one='PSD Model',
        label_two='Estimated PSD'
    )
    fig_model.savefig(f'plots/signal_{i}_forecast.pdf')
    fig_model.show()
    # fig_model.savefig(f'{paper_plots_dir}/psd_model/figure_X_signal_{i}.png', dpi=300)

In [ ]:
for i, truth_time_s in enumerate(truth_times_s):
    fig_model, _ = plotting_utils.plot_around_time(
        results_one=results_model_filtered,
        results_two=results_estimate_filtered,
        signal_truth_time=truth_time_s,
        truth_times=truth_times_s,
        start_time_offset=-20,
        end_time_offset=20,
        times_before=times_before,
        signal_number=i,
        label_one='PSD Model',
        label_two='Estimated PSD',
        predicted_time=False,
    )
    fig_model.show()
    fig_model.savefig(f'plots/signal_{i}_end_time.pdf')
    # fig_model.savefig(f'{paper_plots_dir}/psd_model/figure_X_signal_{i}.png', dpi=300)

In [ ]:
# For signal zero - plot the results filtered so that the predicted time is within an hour of the merger
example_signal = 0

# Get the zero-latency results for this signal
results_zero_latency = get_results_zero_latency(
    "../zero_latency/results/psd_estimate/data_runs_raw_{time_before}_results.hdf",
    times_before
)

# Filter the zero latency results to only use the closest point for each
import numpy as np

zl_results_data_end_time = []
zl_results_snr = []

for t_before in times_before:
    this_t = results_zero_latency['time_before'] == t_before
    closest_match = np.argmin(
        abs(results_zero_latency['time'][this_t] - truth_times_s[example_signal])
    )
    zl_results_data_end_time.append(results_zero_latency['data_end_time'][this_t][closest_match])
    zl_results_snr.append(results_zero_latency['snr'][this_t][closest_match])

zl_results_data_end_time = np.array(zl_results_data_end_time)
zl_results_snr = np.array(zl_results_snr)

width = plt.rcParams["figure.figsize"][0] * 1.25
height = plt.rcParams["figure.figsize"][1]

fig, ax = plt.subplots(1, figsize=(width, height),)


ip_times_before = np.unique(results_estimate_all['time_before'])
ip_results_data_end_time = []
ip_results_snr = []
for t_before in ip_times_before:
    this_t = results_estimate_all['time_before'] == t_before
    closest_match = np.argmin(
        abs(results_estimate_all['time'][this_t] - truth_times_s[example_signal])
    )
    ip_results_data_end_time.append(results_estimate_all['data_end_time'][this_t][closest_match])
    ip_results_snr.append(results_estimate_all['snr'][this_t][closest_match])

inpainting_gaps_results = get_results(
    "results/data_runs_signal_0_gaps.hdf",
)
ipg_times_before = np.unique(inpainting_gaps_results['time_before'])
ipg_results_data_end_time = []
ipg_results_snr = []
for t_before in ip_times_before:
    this_t = inpainting_gaps_results['time_before'] == t_before
    closest_match = np.argmin(
        abs(inpainting_gaps_results['time'][this_t] - truth_times_s[example_signal])
    )
    ipg_results_data_end_time.append(inpainting_gaps_results['data_end_time'][this_t][closest_match])
    ipg_results_snr.append(inpainting_gaps_results['snr'][this_t][closest_match])


ax.plot(
    (ip_results_data_end_time - truth_times_s[example_signal]) / 86400,
    ip_results_snr,
    linewidth=1.25,
    color='tab:blue',
    label='Inpainting Results',
    zorder=10
)
ax.plot(
    (ipg_results_data_end_time - truth_times_s[example_signal]) / 86400,
    ipg_results_snr,
    linewidth=1.25,
    color='tab:orange',
    linestyle='--',
    label='Inpainting with Gaps',
    zorder=8
)

ax.scatter(
    (zl_results_data_end_time - truth_times_s[example_signal]) / 86400,
    zl_results_snr,
    edgecolor='k',
    marker='o',
    facecolors='none',
    label='Zero Latency Results',
    zorder=20

)

norm = mcolors.LogNorm(vmin=min(times_before), vmax=max(times_before))
cmap = cm.get_cmap('rainbow')
sm = cm.ScalarMappable(norm=norm, cmap=cmap)

for time_before in times_before:
    ax.axvline(
        -time_before,
        c=sm.to_rgba(time_before),
        linestyle=':',
        linewidth=1.25,
        zorder=5
    )

for gap in [(2.5, 1.5), (5.5, 4.5), (10.5, 9.5)]:
    ax.axvspan(-gap[0], -gap[1], color='lightgray', zorder=0)
ax.axvspan(-np.inf, -np.inf, color='lightgray', zorder=0, label='Data gaps')

ax.grid(zorder=0)
ax.legend(loc='upper left')
ax.set_ylabel('SNR')
ax.set_xlabel('Data End Time offset (days)')
fig.show()
fig.savefig(f'plots/signal_{example_signal}_end_timeline.pdf')
fig.savefig(f'{paper_plots_dir}/figure_X_signal_{example_signal}_timeline.png', dpi=300)